In [314]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
import numpy as np
import os

## Import the data

In [315]:
clinical_data_path = '/media/shadab/Others/cirrhosis+patient+survival+prediction+dataset-1/'
clinical_data = pd.read_csv('/media/shadab/Others/cirrhosis+patient+survival+prediction+dataset-1/cirrhosis.csv')

In [316]:
clinical_data.shape

(418, 20)

In [317]:
clinical_data.head(20)

,ID,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,1,400,D,D-penicillamine,21464,F,Y,Y,Y,Y,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,2,4500,C,D-penicillamine,20617,F,N,Y,Y,N,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,3,1012,D,D-penicillamine,25594,M,N,N,N,S,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,4,1925,D,D-penicillamine,19994,F,N,Y,Y,S,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,5,1504,CL,Placebo,13918,F,N,Y,Y,N,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0
5,6,2503,D,Placebo,24201,F,N,Y,N,N,0.8,248.0,3.98,50.0,944.0,93.00,63.0,NaN,11.0,3.0
6,7,1832,C,Placebo,20284,F,N,Y,N,N,1.0,322.0,4.09,52.0,824.0,60.45,213.0,204.0,9.7,3.0
7,8,2466,D,Placebo,19379,F,N,N,N,N,0.3,280.0,4.00,52.0,4651.2,28.38,189.0,373.0,11.0,3.0
8,9,2400,D,D-penicillamine,15526,F,N,N,Y,N,3.2,562.0,3.08,79.0,2276.0,144.15,88.0,251.0,11.0,2.0
9,10,51,D,Placebo,25772,F,Y,N,Y,Y,12.6,200.0,2.74,140.0,918.0,147.25,143.0,302.0,11.5,4.0


In [318]:
# Find the missing values in each column
clinical_data.isnull().sum()

ID                 0
N_Days             0
Status             0
Drug             106
Age                0
Sex                0
Ascites          106
Hepatomegaly     106
Spiders          106
Edema              0
Bilirubin          0
Cholesterol      134
Albumin            0
Copper           108
Alk_Phos         106
SGOT             106
Tryglicerides    136
Platelets         11
Prothrombin        2
Stage              6
dtype: int64

In [319]:
# Find the unique values in Drug column
clinical_data['Drug'].unique()

array(['D-penicillamine', 'Placebo', nan], dtype=object)

In [320]:
# Find the range of values in numberical columns
numerical_columns = clinical_data.select_dtypes(include=['int64', 'float64']).columns
for column in numerical_columns:
    print(f"Statistics for column: {column}")
    print(f"min={clinical_data[column].min()}, max={clinical_data[column].max()}")
    print(f"mean={clinical_data[column].mean()}, std={clinical_data[column].std()}")

Statistics for column: ID
min=1, max=418
mean=209.5, std=120.81045760473994
Statistics for column: N_Days
min=41, max=4795
mean=1917.7822966507176, std=1104.6729923907321
Statistics for column: Age
min=9598, max=28650
mean=18533.351674641148, std=3815.8450545514697
Statistics for column: Bilirubin
min=0.3, max=28.0
mean=3.2208133971291866, std=4.407506384141372
Statistics for column: Cholesterol
min=120.0, max=1775.0
mean=369.51056338028167, std=231.944545037874
Statistics for column: Albumin
min=1.96, max=4.64
mean=3.4974401913875592, std=0.4249716057796193
Statistics for column: Copper
min=4.0, max=588.0
mean=97.64838709677419, std=85.61391990897141
Statistics for column: Alk_Phos
min=289.0, max=13862.4
mean=1982.6557692307692, std=2140.388824451761
Statistics for column: SGOT
min=26.35, max=457.25
mean=122.55634615384616, std=56.699524863313016
Statistics for column: Tryglicerides
min=33.0, max=598.0
mean=124.70212765957447, std=65.14863866583947
Statistics for column: Platelets
min

In [321]:
# Find the unique values in columns Sex	Ascites	Hepatomegaly	Spiders	Edema
categorical_columns = ['Sex', 'Ascites', 'Hepatomegaly', 'Spiders', 'Edema']
for column in categorical_columns:
    print(f"Unique values in column {column}: {clinical_data[column].unique()}")

Unique values in column Sex: ['F' 'M']
Unique values in column Ascites: ['Y' 'N' nan]
Unique values in column Hepatomegaly: ['Y' 'N' nan]
Unique values in column Spiders: ['Y' 'N' nan]
Unique values in column Edema: ['Y' 'N' 'S']


In [322]:
# drop the id column
clinical_data = clinical_data.drop(columns=['ID'])

In [323]:
# drop the rows that has missing values in target column 'Stage'
clinical_data = clinical_data.dropna(subset=['Stage'])

In [324]:
clinical_data.shape

(412, 19)

## One hot encoding for categorical columns with non-missing values

For drug column, impute missing values with 'None'

In [325]:
# For drug column, impute missing values with 'None'
clinical_data['Drug'] = clinical_data['Drug'].fillna('None')

In [326]:
non_missing_categorical = ['Status', 'Drug', 'Sex', 'Edema']

# keep only columns that actually exist in the dataframe
cols_to_encode = [c for c in non_missing_categorical if c in clinical_data.columns]

if not cols_to_encode:
    print("No matching categorical columns found to encode:", non_missing_categorical)
else:
    for c in cols_to_encode:
        # ensure strings and mark missing explicitly
        clinical_data[c] = clinical_data[c].astype('object').fillna('Missing')

        # label encode -> integers 0..k-1, then shift to 1..k if you prefer 1-based encoding
        le = LabelEncoder()
        encoded = le.fit_transform(clinical_data[c])  # numpy array of ints

        # create a pandas Series with nullable integer dtype and preserve the original index
        clinical_data[c] = pd.Series(encoded + 1, index=clinical_data.index, dtype="Int64")

        # show mapping: category -> integer
        mapping = {cat: idx + 1 for idx, cat in enumerate(le.classes_)}
        print(f"{c} mapping: {mapping}")

Status mapping: {'C': 1, 'CL': 2, 'D': 3}
Drug mapping: {'D-penicillamine': 1, 'None': 2, 'Placebo': 3}
Sex mapping: {'F': 1, 'M': 2}
Edema mapping: {'N': 1, 'S': 2, 'Y': 3}


In [327]:
clinical_data.head(20)

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,400,3,1,21464,1,Y,Y,Y,3,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,4500,1,1,20617,1,N,Y,Y,1,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,1012,3,1,25594,2,N,N,N,2,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,1925,3,1,19994,1,N,Y,Y,2,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,1504,2,3,13918,1,N,Y,Y,1,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0
5,2503,3,3,24201,1,N,Y,N,1,0.8,248.0,3.98,50.0,944.0,93.00,63.0,NaN,11.0,3.0
6,1832,1,3,20284,1,N,Y,N,1,1.0,322.0,4.09,52.0,824.0,60.45,213.0,204.0,9.7,3.0
7,2466,3,3,19379,1,N,N,N,1,0.3,280.0,4.00,52.0,4651.2,28.38,189.0,373.0,11.0,3.0
8,2400,3,1,15526,1,N,N,Y,1,3.2,562.0,3.08,79.0,2276.0,144.15,88.0,251.0,11.0,2.0
9,51,3,3,25772,1,Y,N,Y,3,12.6,200.0,2.74,140.0,918.0,147.25,143.0,302.0,11.5,4.0


## Handling Missing Values

### Numerical Columns

- Numerical Columns: Cholesterol Copper	Alk_Phos SGOT Tryglicerides	Platelets	Prothrombin
- Temporarily fill categorical missing values with 'Missing' for cols: Ascites, Hepatomegaly, Spiders
- Use KNN imputer to fill numerical missing values
- Restore categorical missing values back to NaN
- For other non missing categorical columns, convert to numerical using one-hot encoding, then revert back to categorical

In [328]:

numerical_cols = ['Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
numerical_cols = [c for c in numerical_cols if c in clinical_data.columns]

# Categorical columns that may have missing values
cat_missing_cols = [c for c in ['Ascites', 'Hepatomegaly', 'Spiders'] if c in clinical_data.columns]

# Non-missing categorical predictors (already label-encoded)
non_missing_categorical = ['Status', 'Drug', 'Sex', 'Edema']
cols_to_include = [c for c in non_missing_categorical if c in clinical_data.columns]

# Additional predictors
additional_numeric = ['N_Days', 'Age', 'Bilirubin', 'Albumin', 'Stage']
predictor_cols = [c for c in numerical_cols + cols_to_include + cat_missing_cols + additional_numeric
                  if c in clinical_data.columns]

# ------------------- Imputation -------------------
if not numerical_cols:
    print("No numerical columns found for KNN imputation; skipping numeric imputation.")
else:
    # 1) Save mask of original missingness for categorical columns
    original_missing_mask = {c: clinical_data[c].isnull() for c in cat_missing_cols}

    # 2) Temporarily fill categorical missing values with numeric codes for KNN
    for c in cat_missing_cols:
        clinical_data[c] = clinical_data[c].astype('category').cat.codes.replace(-1, np.nan).fillna(0)

    # 3) Prepare temporary dataframe for KNN imputation
    temp_df = clinical_data[predictor_cols].astype(float)

    # Ensure n_neighbors is valid
    n_samples = temp_df.shape[0]
    n_neighbors = min(5, max(1, n_samples - 1))

    # 4) Scale → KNN Impute → Inverse scale
    scaler = StandardScaler()
    temp_scaled = scaler.fit_transform(temp_df.values)

    imputer = KNNImputer(n_neighbors=n_neighbors)
    imputed_scaled = imputer.fit_transform(temp_scaled)

    imputed = scaler.inverse_transform(imputed_scaled)
    imputed_df = pd.DataFrame(imputed, columns=temp_df.columns, index=temp_df.index)

    # 5) Replace numeric columns with imputed values
    clinical_data.loc[:, numerical_cols] = imputed_df[numerical_cols].values

    # 6) Restore original missing categorical values
    for c in cat_missing_cols:
        if c in clinical_data.columns:
            clinical_data.loc[original_missing_mask[c], c] = np.nan

    print("✅ KNN imputation completed for numerical columns:", numerical_cols)
    print("🔹 Predictors used:", predictor_cols)


✅ KNN imputation completed for numerical columns: ['Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin']
🔹 Predictors used: ['Cholesterol', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Status', 'Drug', 'Sex', 'Edema', 'Ascites', 'Hepatomegaly', 'Spiders', 'N_Days', 'Age', 'Bilirubin', 'Albumin', 'Stage']


In [329]:
clinical_data.head(20)

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,Cholesterol,Albumin,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage
0,400,3,1,21464,1,1.0,1.0,1.0,3,14.5,261.0,2.60,156.0,1718.0,137.95,172.0,190.0,12.2,4.0
1,4500,1,1,20617,1,0.0,1.0,1.0,1,1.1,302.0,4.14,54.0,7394.8,113.52,88.0,221.0,10.6,3.0
2,1012,3,1,25594,2,0.0,0.0,0.0,2,1.4,176.0,3.48,210.0,516.0,96.10,55.0,151.0,12.0,4.0
3,1925,3,1,19994,1,0.0,1.0,1.0,2,1.8,244.0,2.54,64.0,6121.8,60.63,92.0,183.0,10.3,4.0
4,1504,2,3,13918,1,0.0,1.0,1.0,1,3.4,279.0,3.53,143.0,671.0,113.15,72.0,136.0,10.9,3.0
5,2503,3,3,24201,1,0.0,1.0,0.0,1,0.8,248.0,3.98,50.0,944.0,93.00,63.0,240.0,11.0,3.0
6,1832,1,3,20284,1,0.0,1.0,0.0,1,1.0,322.0,4.09,52.0,824.0,60.45,213.0,204.0,9.7,3.0
7,2466,3,3,19379,1,0.0,0.0,0.0,1,0.3,280.0,4.00,52.0,4651.2,28.38,189.0,373.0,11.0,3.0
8,2400,3,1,15526,1,0.0,0.0,1.0,1,3.2,562.0,3.08,79.0,2276.0,144.15,88.0,251.0,11.0,2.0
9,51,3,3,25772,1,1.0,0.0,1.0,3,12.6,200.0,2.74,140.0,918.0,147.25,143.0,302.0,11.5,4.0


In [330]:
clinical_data.isnull().sum()

N_Days             0
Status             0
Drug               0
Age                0
Sex                0
Ascites          100
Hepatomegaly     100
Spiders          100
Edema              0
Bilirubin          0
Cholesterol        0
Albumin            0
Copper             0
Alk_Phos           0
SGOT               0
Tryglicerides      0
Platelets          0
Prothrombin        0
Stage              0
dtype: int64

### Categorical Columns

- The columns with missing values are: Ascites, Hepatomegaly, Spiders
- Create 3 new columns to indicate whether the value was missing in these columns originally
- Impute the missing values with KNN

In [331]:
# Columns with missing categorical values
cat_missing_cols = ['Ascites', 'Hepatomegaly', 'Spiders']
cat_missing_cols = [c for c in cat_missing_cols if c in clinical_data.columns]

# ------------------- Prepare categorical columns -------------------
for c in cat_missing_cols:
    # 1) Create missing indicator column
    clinical_data[f"{c}_missing"] = clinical_data[c].isna().astype(int)  # 1 = missing, 0 = not missing

    # 2) Convert categorical values to integer codes, temporarily fill missing with np.nan
    clinical_data[c] = clinical_data[c].astype('category')
    clinical_data[c] = clinical_data[c].cat.codes.replace(-1, np.nan)  # KNNImputer requires numeric

# ------------------- Prepare predictors for KNN -------------------
# Use all columns as predictors
predictor_cols = [c for c in clinical_data.columns if c not in cat_missing_cols]  # everything else
knn_predictors = clinical_data[predictor_cols].copy()

# Ensure categorical columns are converted to numeric codes
for col in knn_predictors.select_dtypes('category').columns:
    knn_predictors[col] = knn_predictors[col].cat.codes.replace(-1, np.nan)

# Combine predictors + columns to impute
knn_df = pd.concat([knn_predictors, clinical_data[cat_missing_cols]], axis=1).astype(float)

# Print the predictors being used
print("Columns used as predictors for KNN imputation:", list(knn_predictors.columns))

# ------------------- KNN Imputation -------------------
n_samples = knn_df.shape[0]
n_neighbors = min(5, max(1, n_samples - 1))

imputer = KNNImputer(n_neighbors=n_neighbors)
imputed = imputer.fit_transform(knn_df)
imputed_df = pd.DataFrame(imputed, columns=knn_df.columns, index=knn_df.index)

# ------------------- Assign imputed values back -------------------
for c in cat_missing_cols:
    # Round to nearest integer, then shift to 1-based codes
    clinical_data[c] = np.rint(imputed_df[c]).astype(int) + 1

    # Optional: verify unique values after imputation
    print(f"{c} imputed unique values:", sorted(clinical_data[c].unique()))

# The missing indicator columns already exist: 1 = originally missing, 0 = not missing


Columns used as predictors for KNN imputation: ['N_Days', 'Status', 'Drug', 'Age', 'Sex', 'Edema', 'Bilirubin', 'Cholesterol', 'Albumin', 'Copper', 'Alk_Phos', 'SGOT', 'Tryglicerides', 'Platelets', 'Prothrombin', 'Stage', 'Ascites_missing', 'Hepatomegaly_missing', 'Spiders_missing']
Ascites imputed unique values: [np.int64(1), np.int64(2)]
Hepatomegaly imputed unique values: [np.int64(1), np.int64(2)]
Spiders imputed unique values: [np.int64(1), np.int64(2)]


In [332]:
clinical_data.head(50)

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,...,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Ascites_missing,Hepatomegaly_missing,Spiders_missing
0,400,3,1,21464,1,2,2,2,3,14.5,...,156.0,1718.0,137.95,172.0,190.0,12.2,4.0,0,0,0
1,4500,1,1,20617,1,1,2,2,1,1.1,...,54.0,7394.8,113.52,88.0,221.0,10.6,3.0,0,0,0
2,1012,3,1,25594,2,1,1,1,2,1.4,...,210.0,516.0,96.10,55.0,151.0,12.0,4.0,0,0,0
3,1925,3,1,19994,1,1,2,2,2,1.8,...,64.0,6121.8,60.63,92.0,183.0,10.3,4.0,0,0,0
4,1504,2,3,13918,1,1,2,2,1,3.4,...,143.0,671.0,113.15,72.0,136.0,10.9,3.0,0,0,0
5,2503,3,3,24201,1,1,2,1,1,0.8,...,50.0,944.0,93.00,63.0,240.0,11.0,3.0,0,0,0
6,1832,1,3,20284,1,1,2,1,1,1.0,...,52.0,824.0,60.45,213.0,204.0,9.7,3.0,0,0,0
7,2466,3,3,19379,1,1,1,1,1,0.3,...,52.0,4651.2,28.38,189.0,373.0,11.0,3.0,0,0,0
8,2400,3,1,15526,1,1,1,2,1,3.2,...,79.0,2276.0,144.15,88.0,251.0,11.0,2.0,0,0,0
9,51,3,3,25772,1,2,1,2,3,12.6,...,140.0,918.0,147.25,143.0,302.0,11.5,4.0,0,0,0


In [333]:
clinical_data.isnull().sum()

N_Days                  0
Status                  0
Drug                    0
Age                     0
Sex                     0
Ascites                 0
Hepatomegaly            0
Spiders                 0
Edema                   0
Bilirubin               0
Cholesterol             0
Albumin                 0
Copper                  0
Alk_Phos                0
SGOT                    0
Tryglicerides           0
Platelets               0
Prothrombin             0
Stage                   0
Ascites_missing         0
Hepatomegaly_missing    0
Spiders_missing         0
dtype: int64

In [334]:
# print unique values in the columns: Ascites_missing	Hepatomegaly_missing	Spiders_missing
print("Ascites_missing unique values:", clinical_data['Ascites_missing'].unique())
print("Hepatomegaly_missing unique values:", clinical_data['Hepatomegaly_missing'].unique())
print("Spiders_missing unique values:", clinical_data['Spiders_missing'].unique())

Ascites_missing unique values: [0 1]
Hepatomegaly_missing unique values: [0 1]
Spiders_missing unique values: [0 1]


## Feature Scaling

In [335]:
# ---------------- Numeric columns ----------------
# Large-range numeric columns
large_range_cols = ['N_Days', 'Age', 'Cholesterol', 'Copper', 'Alk_Phos', 
                    'SGOT', 'Tryglicerides', 'Platelets', 'Bilirubin']

# Small-range numeric columns
small_range_cols = ['Albumin', 'Prothrombin']

# ---------------- Scaling ----------------
scaled_data = clinical_data.copy()

# Function to scale to custom range [min_val, max_val]
def custom_scale(series, target_min, target_max):
    s_min, s_max = series.min(), series.max()
    return (series - s_min) / (s_max - s_min) * (target_max - target_min) + target_min

# Scale large-range variables to roughly 1-200
for col in large_range_cols:
    if col in scaled_data.columns:
        scaled_data[col] = custom_scale(scaled_data[col], 1, 200)

# Scale small-range variables to roughly 1-10
for col in small_range_cols:
    if col in scaled_data.columns:
        scaled_data[col] = custom_scale(scaled_data[col], 1, 10)

# Optional: check results
for col in large_range_cols + small_range_cols:
    if col in scaled_data.columns:
        print(f"{col}: min={scaled_data[col].min():.2f}, max={scaled_data[col].max():.2f}")

# scaled_data now contains all numeric columns scaled appropriately

clinical_data = scaled_data


N_Days: min=1.00, max=200.00
Age: min=1.00, max=200.00
Cholesterol: min=1.00, max=200.00
Copper: min=1.00, max=200.00
Alk_Phos: min=1.00, max=200.00
SGOT: min=1.00, max=200.00
Tryglicerides: min=1.00, max=200.00
Platelets: min=1.00, max=200.00
Bilirubin: min=1.00, max=200.00
Albumin: min=1.00, max=10.00
Prothrombin: min=1.00, max=10.00


In [336]:
# Find the range of values in numberical columns
numerical_columns = clinical_data.select_dtypes(include=['int64', 'float64']).columns
for column in numerical_columns:
    print(f"Statistics for column: {column}")
    print(f"min={clinical_data[column].min()}, max={clinical_data[column].max()}")
    print(f"mean={clinical_data[column].mean()}, std={clinical_data[column].std()}")

Statistics for column: N_Days
min=1.0, max=200.0
mean=79.5215898926198, std=46.04297184621493
Statistics for column: Status
min=1, max=3
mean=1.8228155339805825, std=0.9540100204789083
Statistics for column: Drug
min=1, max=3
mean=1.9902912621359223, std=0.8712230475935869
Statistics for column: Age
min=1.0, max=200.0
mean=93.96436044224393, std=39.93897981416482
Statistics for column: Sex
min=1, max=2
mean=1.1067961165048543, std=0.30922936500198195
Statistics for column: Ascites
min=1, max=2
mean=1.0606796116504855, std=0.2390319574107106
Statistics for column: Hepatomegaly
min=1, max=2
mean=1.5436893203883495, std=0.49869316379228845
Statistics for column: Spiders
min=1, max=2
mean=1.2742718446601942, std=0.44668904023948053
Statistics for column: Edema
min=1, max=3
mean=1.203883495145631, std=0.5099348424761827
Statistics for column: Bilirubin
min=1.0, max=200.0
mean=22.032718797097893, std=31.821470056129183
Statistics for column: Cholesterol
min=1.0, max=200.0
mean=31.05090869093

In [337]:
clinical_data.head(50)

,N_Days,Status,Drug,Age,Sex,Ascites,Hepatomegaly,Spiders,Edema,Bilirubin,...,Copper,Alk_Phos,SGOT,Tryglicerides,Platelets,Prothrombin,Stage,Ascites_missing,Hepatomegaly_missing,Spiders_missing
0,16.027556,3,1,124.941528,1,2,2,2,3,103.014440,...,52.794521,21.950609,52.539568,49.957522,51.842315,4.2,4.0,0,0,0
1,187.651451,1,1,116.094531,1,1,2,2,1,6.747292,...,18.037671,105.178334,41.257206,20.371681,64.155689,2.6,3.0,0,0,0
2,41.645562,3,1,168.079782,2,1,1,1,2,8.902527,...,71.195205,4.328053,33.212230,8.748673,36.351297,4.0,4.0,0,0,0
3,79.863273,3,1,109.587235,1,1,2,2,2,11.776173,...,21.445205,86.514845,16.831330,21.780531,49.061876,2.3,4.0,0,0,0
4,62.240429,2,3,46.122822,1,1,2,2,1,23.270758,...,48.364726,6.600513,41.086331,14.736283,30.393214,2.9,3.0,0,0,0
5,104.058056,3,3,153.529761,1,1,2,1,1,4.592058,...,16.674658,10.602973,31.780576,11.566372,71.702595,3.0,3.0,0,0,0
6,75.970341,1,3,112.616313,1,1,2,1,1,6.028881,...,17.356164,8.843650,16.748201,64.398230,57.403194,1.7,3.0,0,0,0
7,102.509255,3,3,103.163500,1,1,1,1,1,1.000000,...,17.356164,64.954337,1.937503,55.945133,124.530938,3.0,3.0,0,0,0
8,99.746529,3,1,62.918539,1,1,1,2,1,21.833935,...,26.556507,30.131463,55.402878,20.371681,76.071856,3.0,2.0,0,0,0
9,1.418595,3,3,169.939009,1,2,1,2,3,89.364621,...,47.342466,10.221787,56.834532,39.743363,96.329341,3.5,4.0,0,0,0


## Export the preprocessed data

In [338]:
# export the preprocessed data
clinical_data.to_csv(os.path.join(clinical_data_path, 'clinical_data_processed.csv'), index=False)